# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print overview
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished} | Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Keywords: {', '.join(metadata.keywords)}")
print(f"Dataset @id: {metadata.id}")


## 2. Data Overview
Review available record sets, fields, and their unique `@id` identifiers. This information is necessary for later referencing and extraction.

**Note:** All Croissant entities (record sets, fields, columns) are referenced by their `@id` fields.

In [ ]:
record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}")

# Display the @id and summary of each record set
for rs in record_sets:
    print(f"Record Set @id: {rs.id}")
    print(f"  Record Set Name: {getattr(rs, 'name', 'N/A')}")
    print(f"  Description: {getattr(rs, 'description', 'N/A')}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, Name: {getattr(field, 'name', 'N/A')}, DataType: {getattr(field, 'dataType', 'N/A')}")
    print("---")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview above.

**Tip:** You can inspect field @ids via the overview above and reference them accordingly.

In [ ]:
# Extract all record sets as DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Preview column names and first few rows from the main record set
# For demonstration, choose the first record_set.
main_record_set_id = record_set_ids[0]
print(f"Columns in record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())

print(f"Sample data:")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
We use field and record set `@id`s throughout.

In [ ]:
# Identify numeric fields
main_df = dataframes[main_record_set_id]
numeric_fields = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
print(f"Numeric fields: {numeric_fields}")

# Example: filter on PHQ-9 total score (commonly named phq9_total, or a similar column)
# Replace with exact field @id as found in the columns above
phq9_field_id = None
for col in main_df.columns:
    if 'phq' in col.lower() and ('total' in col.lower() or 'score' in col.lower()):
        phq9_field_id = col
        break

if phq9_field_id:
    threshold = 10
    filtered_df = main_df[main_df[phq9_field_id] > threshold]
    print(f"Filtered records with {phq9_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the PHQ-9 total score
    filtered_df[f"{phq9_field_id}_normalized"] = (
        filtered_df[phq9_field_id] - filtered_df[phq9_field_id].mean()
    ) / filtered_df[phq9_field_id].std()
    print(f"Normalized {phq9_field_id} for filtered records:")
    display(filtered_df[[phq9_field_id, f"{phq9_field_id}_normalized"]].head())

    # Group by education level (e.g., level_of_education @id or similar column)
    education_field = None
    for col in main_df.columns:
        if 'education' in col.lower():
            education_field = col
            break

    if education_field and education_field in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(education_field)[phq9_field_id].mean().reset_index()
        )
        print(f"Mean {phq9_field_id} grouped by {education_field}:")
        display(grouped_df.head())
else:
    print("PHQ-9 score field not found. Please check the field @id from data overview.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
Use field `@id` references for axes and legends.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Distribution of PHQ-9 scores
if phq9_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[phq9_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {phq9_field_id}")
    plt.xlabel(phq9_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by education level
    if education_field and education_field in main_df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=education_field, y=phq9_field_id, data=main_df)
        plt.title(f"{phq9_field_id} by {education_field}")
        plt.xlabel(education_field)
        plt.ylabel(phq9_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Cannot plot PHQ-9 score distribution - field not found.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded the dataset and reviewed its metadata and available record sets via their `@id`s.
- Extracted and previewed survey records, exploring both demographic and mental health score fields.
- Applied simple filtering and normalization for PHQ-9 scores, grouped by education to illustrate deeper analysis.
- Visualized key distributions, showing how the dataset can be used to highlight mental health patterns in Kilifi County.

**Next Steps:**
- Explore other assessment scores such as GAD-7 or PSQ with their respective field `@id`s.
- Investigate correlations between demographic variables and mental health indicators.
- Prepare data for machine learning or statistical modeling using `mlcroissant` and pandas.